In [50]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [51]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.set_printoptions(suppress=True)

In [52]:
from simulators import NestedModelFamily, ContextManager
from simulators.benchmarks import DDM, RDM, CDM
from adapters import Adapter

# Metas

In [53]:
ddm_intrinsics = ["v", "a", "tau", "s_v", "s_tau", "decay"]

# Priors

In [54]:
ddm_priors = {
    "v":     {"intercept": lambda: np.random.gamma(3.0, 0.8),
              "slope":     lambda: np.random.normal(0.0, 3.0)},
    "a":     {"intercept": lambda: np.random.gamma(10.0, 0.3),
              "slope":     lambda: np.random.normal(0.0, 1.0)},
    "tau":   {"intercept": lambda: np.random.gamma(3.0, 0.2),
              "slope":     lambda: 0.0},
    "s_v":   {"intercept": lambda: np.random.gamma(1.0, 0.2),
              "slope":     lambda: 0.0},
    "s_tau": {"intercept": lambda: np.random.uniform(0.0, 0.4),
              "slope":     lambda: 0.0},
    "decay": {"intercept": lambda: np.random.gamma(1.0, 0.4),
              "slope":     lambda: 0.0},
}

# Context Manager

In [55]:
context_manager = ContextManager()

# Model Family

In [56]:
model_family = NestedModelFamily(
    name="DDM",
    model=DDM(),
    context_manager=context_manager,
    prior_fun=ddm_priors,
    intrinsic_params=ddm_intrinsics,
)

In [57]:
samples = model_family.batch_sample(
    batch_size=10,
    mask_randomizer_kwargs=dict(
        free_intrinsics={"v", "a", "tau", "s_v", "decay"},
        fixed_intrinsics={"s_tau"}
    ),
    min_num_obs=20,
    max_num_obs=500,
    flatten_param_outputs=False
)

(30, 6) (30, 6)
(0, 6) (0, 6)
(21, 6) (21, 6)
(3, 6) (3, 6)
(9, 6) (9, 6)
(15, 6) (15, 6)
(12, 6) (12, 6)
(21, 6) (21, 6)
(15, 6) (15, 6)
(21, 6) (21, 6)


In [58]:
samples["param_masks"].shape

(10, 30, 6)

In [59]:
samples["param_matrices"].shape

(10, 30, 6)

In [60]:
samples["regressor_masks"].shape

(10, 30)

In [61]:
samples

{'model_names': ['DDM',
  'DDM',
  'DDM',
  'DDM',
  'DDM',
  'DDM',
  'DDM',
  'DDM',
  'DDM',
  'DDM'],
 'design_configs': [{'u_0': ['v', 'a'],
   'u_1': ['a', 's_v', 'decay'],
   'u_2': ['v', 'a', 'tau', 's_v', 'decay'],
   'u_3': ['a', 'tau', 's_v'],
   'u_4': [],
   'u_5': ['tau', 's_v'],
   'u_6': ['v', 'a', 'tau'],
   'u_7': ['v', 'a'],
   'u_8': ['v', 'a', 's_v', 'decay'],
   'u_9': ['v', 'a', 'tau', 'decay']},
  {},
  {'u_0': ['a', 'tau'],
   'u_1': ['v', 'a', 'tau', 's_v'],
   'u_2': ['v', 'a', 'tau', 's_v'],
   'u_3': ['a', 'tau', 's_v', 'decay'],
   'u_4': ['tau', 's_v'],
   'u_5': ['tau'],
   'u_6': ['v', 'a', 'tau', 'decay']},
  {'u_0': ['v']},
  {'u_0': ['v', 'a'], 'u_1': ['v', 's_v', 'decay'], 'u_2': ['v', 'a', 's_v']},
  {'u_0': ['a', 's_v'],
   'u_1': ['s_v', 'decay'],
   'u_2': ['a', 's_v', 'decay'],
   'u_3': ['tau', 's_v'],
   'u_4': ['v', 's_v', 'decay']},
  {'u_0': ['a', 'tau'],
   'u_1': ['a', 'decay'],
   'u_2': ['v'],
   'u_3': ['a', 's_v', 'decay']},
  {'u_0'

# Adapter

In [62]:
adapter = Adapter()

In [63]:
design_matrices = adapter.convert_dtype(samples["design_matrices"], dtype=np.float32)
param_masks = adapter.convert_dtype(samples["param_masks"], dtype=np.float32)
rts = adapter.convert_dtype(samples["sim_data"]["rts"], dtype=np.float32)
choices = adapter.convert_dtype(samples["sim_data"]["choices"], dtype=np.float32)

/home/a_huangm13/Documents/Research/Projects/BayesGPT/bayesgpt/adapters/adapter.py:123: RuntimeWarning: overflow encountered in cast
  return a.astype(dtype, copy=copy, casting="same_kind")
/home/a_huangm13/Documents/Research/Projects/BayesGPT/bayesgpt/adapters/adapter.py:123: RuntimeWarning: invalid value encountered in cast
  return a.astype(dtype, copy=copy, casting="same_kind")


In [64]:
batch_size, num_obs, num_cols = design_matrices.shape
print(batch_size, num_obs, num_cols)

10 495 30


In [65]:
y_rts_col = adapter.atleast_2d(rts, orientation="col")                 # (N, 1)
y_ch_col  = adapter.atleast_2d(rts,  orientation="col")

In [66]:
sim_data = adapter.concatenate([y_rts_col, y_ch_col], axis=1, dtype=np.float32, pad=False)

# RDM

In [67]:
rdm_priors = {
    "v":      {"intercept": lambda: np.random.gamma(3.0, 0.8),
               "slope":     lambda: np.random.normal(0.0, 3.0)},
    "a":      {"intercept": lambda: np.random.gamma(10.0, 0.3),
               "slope":     lambda: np.random.normal(0.0, 1.0)},
    "tau":    {"intercept": lambda: np.random.gamma(3.0, 0.2),
               "slope":     lambda: np.random.normal(0.0, 0.2)},
    "decay":  {"intercept": lambda: np.random.gamma(1.0, 0.4),
               "slope":     lambda: np.random.normal(0.0, 0.2)},
}

In [70]:
family = NestedModelFamily(
    name="RDM",
    model=RDM(),
    context_manager=context_manager,
    prior_fun=rdm_priors,
    intrinsic_params=ddm_intrinsics,
)

In [74]:
samples = family.batch_sample(
    batch_size=3,
    mask_randomizer_kwargs=dict(
        free_intrinsics={"v", "a", "tau"},
        fixed_intrinsics={"decay"}
    ),
    min_num_obs=20,
    max_num_obs=500,
    flatten_param_outputs=False,
)

No context provided. Building default context from the model.
# alternatives: 3
No context provided. Building default context from the model.
# alternatives: 2
No context provided. Building default context from the model.
# alternatives: 3
(18, 6) (18, 6)
(0, 6) (0, 6)
(6, 6) (6, 6)


In [75]:
samples

{'model_names': ['RDM', 'RDM', 'RDM'],
 'design_configs': [{'u_0': ['tau'],
   'u_1': ['v', 'a'],
   'u_2': ['v', 'a', 'tau'],
   'u_3': ['v', 'a'],
   'u_4': ['a'],
   'u_5': ['v', 'a', 'tau']},
  {},
  {'u_0': [], 'u_1': ['v', 'a', 'tau']}],
 'design_matrices': array([[[0.90046506, 0.        , 0.        , ..., 0.85219397,
          0.        , 0.        ],
         [0.17607609, 0.        , 0.        , ..., 0.45238462,
          0.        , 0.        ],
         [0.40368032, 0.        , 0.        , ..., 0.37759566,
          0.        , 0.        ],
         ...,
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ]],
 
        [[0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 

# CDM

In [76]:
cdm_intrinsics = ["v_x", "v_y", "a", "tau", "decay"]

In [77]:
cdm_priors = {
    "v_x":   {"intercept": lambda: np.random.normal(0.0, 1.0),
              "slope":     lambda: np.random.normal(0.0, 0.5)},
    "v_y":   {"intercept": lambda: np.random.normal(0.0, 1.0),
              "slope":     lambda: np.random.normal(0.0, 0.5)},
    "a":     {"intercept": lambda: np.random.gamma(10.0, 0.3),
              "slope":     lambda: np.random.normal(0.0, 0.5)},
    "tau":   {"intercept": lambda: np.random.gamma(3.0, 0.2),
              "slope":     lambda: np.random.normal(0.0, 0.1)},
    "decay": {"intercept": lambda: np.random.gamma(1.0, 0.4),
              "slope":     lambda: np.random.normal(0.0, 0.1)},
}

In [78]:
family = NestedModelFamily(
    name="CDM",
    model=CDM(),
    context_manager=context_manager,
    prior_fun=cdm_priors,
    intrinsic_params=cdm_intrinsics,
)

In [79]:
out = family.sample(
    design_config=None,
    num_obs=200,
    num_regressors=3,
    max_num_regressors=6,
    max_num_categories=4,   # dummy blocks up to 3 cols per regressor
    keep_intercept=True,
    discrete_prob=0.5,      # mix of continuous and dummy regressors
)

Theta mode: random_uniform


In [80]:
print("Keys:", out.keys())
print("design_matrix:", out["design_matrix"].shape)
print("param_mask:", out["param_mask"].shape)
print("param_matrix:", out["param_matrix"].shape)
print("sim rts/choices:", out["sim_trials"]["rts"].shape, out["sim_trials"]["choices"].shape)

Keys: dict_keys(['model_name', 'design_config', 'design_matrix', 'param_mask', 'param_matrix', 'sim_trials', 'discrete_mask', 'regressor_mask', 'max_num_regressors', 'keep_intercept'])
design_matrix: (200, 10)
param_mask: (50,)
param_matrix: (50,)
sim rts/choices: (200,) (200,)


In [82]:
batch = family.batch_sample(
    batch_size=4,
    num_obs=None,                 # randomized per item
    num_regressors=None,          # randomized per item
    min_num_obs=100,
    max_num_obs=250,
    min_num_regressors=0,
    max_num_regressors=5,
    max_num_categories=4,
    keep_intercept=True,
    discrete_prob=0.6,
    flatten_param_outputs=False,
)

No context provided. Building default context from the model.
Theta mode: random_uniform
No context provided. Building default context from the model.
Theta mode: random_uniform
No context provided. Building default context from the model.
Theta mode: zeros
No context provided. Building default context from the model.
Theta mode: random_uniform
(16, 5) (16, 5)
(16, 5) (16, 5)
(16, 5) (16, 5)
(13, 5) (13, 5)


In [83]:
print("Batched design_matrices:", batch["design_matrices"].shape)
print("Batched sim rts:", batch["sim_data"]["rts"].shape)
print("Batched sim choices:", batch["sim_data"]["choices"].shape)
print("num_obs per item:", batch["num_obs"].reshape(-1))
print("num_regressors per item:", batch["num_regressors"].reshape(-1))

Batched design_matrices: (4, 212, 16)
Batched sim rts: (4, 212)
Batched sim choices: (4, 212)
num_obs per item: [145. 212. 209. 161.]
num_regressors per item: [5. 5. 5. 4.]
